In [4]:
import numpy as np
import torch as th
import matplotlib.pyplot as plt
import json, glob, os


n_frames = 60
path = "/data/mint/sampling/TPAMI/main_result/ffhq/teaser/"
# blur = 'log=paired+difareli+cs+nodpm+trainset_256_cfg=paired+difareli+cs+nodpm+trainset_256.yaml_blurmap'
blur = 'log=paired+difareli+cs+nodpm+trainset_256_cfg=paired+difareli+cs+nodpm+trainset_256.yaml_blurmap_fix'
dec_c = 'log=paired+difareli+cs+nodpm+trainset_256_cfg=paired+difareli+cs+nodpm+trainset_256.yaml_dec_c'
blur_dec_c = 'log=paired+difareli+cs+nodpm+trainset_256_cfg=paired+difareli+cs+nodpm+trainset_256.yaml_blurmap+dec_c_fix'
pf = '/ema_300000/valid/render_face/reverse_sampling/'
sj_file = "/home/mint/Dev/DiFaReli/difareli-faster/visualize_scripts/TPAMI/main_results/FFHQ_CastShadows/For_Sel/aj_ake_top_candidates.json"

os.makedirs(f'./out/', exist_ok=True)
with open(sj_file, 'r') as f:
    sj = json.load(f)['pair']
    print(sj)

for pid, p in sj.items():
    src = p['src']
    dst = p['dst']
    for folder in [blur, blur_dec_c, dec_c]:
        os.makedirs(f'./out/{folder}/src={src}_dst={dst}/imgs/', exist_ok=True)
        shadm = glob.glob(f'{path}/{folder}/{pf}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/dst_shadm_shad_frame*.png')
        # Copy and rename to 00001.png, 00002.png, ...
        for f in shadm:
            idx = int(f.split('frame')[-1].split('.png')[0])
            if idx == 0: continue
            os.system(f'cp {f} ./out/{folder}/src={src}_dst={dst}/imgs/shadm_{idx:05d}.png')
        res = glob.glob(f'{path}/{folder}/{pf}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/res_frame*.png')
        for f in res:
            idx = int(f.split('frame')[-1].split('.png')[0])
            if idx == 0: continue
            os.system(f'cp {f} ./out/{folder}/src={src}_dst={dst}/imgs/res_{idx:05d}.png')
            
        # Make video using ffmpeg crf 17 24 fps
        os.system(f'ffmpeg -y -framerate 24 -i ./out/{folder}/src={src}_dst={dst}/imgs/shadm_%05d.png -c:v libx264 -crf 17 -pix_fmt yuv420p ./out/{folder}/src={src}_dst={dst}/shadm.mp4')
        os.system(f'ffmpeg -y -framerate 24 -i ./out/{folder}/src={src}_dst={dst}/imgs/res_%05d.png -c:v libx264 -crf 17 -pix_fmt yuv420p ./out/{folder}/src={src}_dst={dst}/res.mp4')
        # Vertical stack the two videos
        os.system(f'ffmpeg -y -i ./out/{folder}/src={src}_dst={dst}/res.mp4 -i ./out/{folder}/src={src}_dst={dst}/shadm.mp4 -filter_complex vstack ./out/{folder}/src={src}_dst={dst}.mp4')
    
    # Stack the three videos horizontally [blur, blur_dec_c, dec_c]
    os.system(f'ffmpeg -y -i ./out/{blur}/src={src}_dst={dst}.mp4 -i ./out/{blur_dec_c}/src={src}_dst={dst}.mp4 -i ./out/{dec_c}/src={src}_dst={dst}.mp4 -filter_complex "[0:v][1:v][2:v]hstack=inputs=3" ./out/out_vids/src={src}_dst={dst}.mp4')


{'pair115': {'src': '61992.jpg', 'dst': '60000.jpg'}, 'pair133': {'src': '69809.jpg', 'dst': '60000.jpg'}, 'pair179': {'src': '65797.jpg', 'dst': '60000.jpg'}, 'pair1791': {'src': '64981.jpg', 'dst': '60000.jpg'}, 'pair281': {'src': '60609.jpg', 'dst': '60000.jpg'}, 'pair287': {'src': '60230.jpg', 'dst': '60000.jpg'}}


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab